# VolumeSpiller Benchmark: Parquet Volume vs Arrow In-Memory

This notebook benchmarks two conversion strategies across increasing data sizes.
Compatible with both **classic clusters** and **Databricks Serverless**.

| Strategy | Spark → Polars | Polars → Spark |
|----------|---------------|----------------|
| **Arrow (traditional)** | `df.toArrow()` → `pl.from_arrow()` | `pl_df.to_pandas()` → `spark.createDataFrame()` |
| **VolumeSpiller** | Write Parquet to UC Volume → `pl.read_parquet()` | `pl_df.write_parquet()` to Volume → `spark.read.parquet()` |

**What to expect:**
- Arrow wins at small sizes (no disk I/O overhead).
- VolumeSpiller wins at large sizes — Spark writes Parquet in parallel across partitions, and the driver never holds the full dataset in Python memory.
- The crossover is typically around **500K–2M rows**, depending on cluster, volume I/O speed, and schema width.

**Note:** Arrow's apparent speed comes at a hidden cost — it collects the entire DataFrame through the driver as a single memory object. `VolumeSpiller` avoids this spike entirely, which matters more for wide or deeply nested schemas.

In [ ]:
import os
import statistics
import sys
import time
import uuid
from pathlib import Path

import polars as pl
from pyspark.sql import functions as F
from pyspark.sql import types as T

repo_root = Path(os.getcwd()).parent
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from databricks_scaffold import VolumeSpiller

# ── Configuration ────────────────────────────────────────────────────────────
CATALOG = "main"
SCHEMA = "default"
VOL_NAME = f"benchmark_vol_{uuid.uuid4().hex[:6]}"
REPEATS = 3  # timing runs per data size (median is reported)
SIZES = [
    1000,
    10_000,
    50_000,
    200_000,
    500_000,
    1_000_000,
    2_000_000,
    4_000_000,
    6_000_000,
    8_000_000,
    10_000_000,
    11_000_000,
]  # rows

try:
    spark  # Already injected on a Databricks cluster
except NameError:
    from databricks.connect import DatabricksSession

    spark = DatabricksSession.builder.getOrCreate()

# Arrow optimisation: already on by default in Serverless; locked configs raise
# AnalysisException there, so we wrap it.
try:
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    print("Arrow enabled via spark.conf")
except Exception:
    print("Arrow config is locked (Serverless) — Arrow is on by default, continuing.")

# IS_DEV = True so teardown() preserves the volume for post-run inspection.
IS_DEV = True
spill = VolumeSpiller(spark, CATALOG, SCHEMA, VOL_NAME)
print(f"Volume ready: {spill.full_name}")

## Data Pre-materialisation

Each test DataFrame is written to a Delta table once before any timing starts.
This is the Serverless-compatible equivalent of `cache().count()` — the data is fully staged on storage so each timed run reads pre-existing Delta files, not a freshly computed `spark.range()`.

The temp tables are dropped in the Cleanup cell at the end.

In [0]:
import random
from datetime import datetime, timezone

RUN_ID = uuid.uuid4().hex[:6]  # ties all temp tables to this benchmark run
temp_tables: dict[int, str] = {}  # size → fully-qualified table name
file_sizes: dict[int, int] = {}  # size → estimated parquet file size in bytes


def make_polars_df(n_rows: int) -> pl.DataFrame:
    """Generate a mixed-type Polars DataFrame with n_rows rows."""
    rng = list(range(n_rows))
    return pl.DataFrame(
        {
            "id": rng,
            "category": [str(i % 10) for i in rng],
            "revenue": [random.random() * 10_000 for _ in rng],
            "is_active": [True] * n_rows,
            "created_at": [datetime(2024, 1, 1, tzinfo=timezone.utc)] * n_rows,
        }
    )


def estimate_parquet_bytes(n_rows: int) -> int:
    """Estimate compressed Parquet file size based on schema.

    Schema breakdown (per row, estimated compressed bytes):
    - id (int64): ~8 bytes
    - category (string, 1 char): ~2 bytes
    - revenue (float64): ~8 bytes
    - is_active (bool): ~1 byte
    - created_at (timestamp): ~8 bytes
    Total per row ≈ 27 bytes, zstd compression ~2x ratio
    """
    bytes_per_row = 27
    compression_ratio = 0.5  # zstd compression factor
    return int(n_rows * bytes_per_row * compression_ratio)


print("Pre-materialising test DataFrames as Delta tables...")
for n in SIZES:
    table_name = f"{CATALOG}.{SCHEMA}.benchmark_spark_{RUN_ID}_{n}"
    (
        spark.range(n)
        .select(
            F.col("id").cast(T.LongType()),
            (F.col("id") % 10).cast(T.StringType()).alias("category"),
            (F.rand(seed=42) * 10_000).alias("revenue"),
            F.lit(True).alias("is_active"),
            F.current_timestamp().alias("created_at"),
        )
        .write.format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    temp_tables[n] = table_name
    file_sizes[n] = estimate_parquet_bytes(n)
    print(f"  {n:>9,} rows → {table_name} (~{file_sizes[n] / 1024 / 1024:.1f} MB estimated)")

print("\nWarm-up: exercising both code paths before timing...")
_warmup_spark = spark.table(temp_tables[SIZES[0]])
_ = _warmup_spark.toArrow()
_ = spill.spark_to_polars(_warmup_spark, cleanup=True)
print("Ready.")

## Benchmark Functions

In [0]:
def _time(fn, repeats: int) -> dict:
    """Run fn() `repeats` times and return timing stats in seconds."""
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return {
        "median": statistics.median(times),
        "min": min(times),
        "max": max(times),
    }


def _format_bytes(num_bytes: int) -> str:
    """Format bytes into human-readable string."""
    for unit in ["B", "KB", "MB", "GB"]:
        if abs(num_bytes) < 1024.0:
            return f"{num_bytes:.1f} {unit}"
        num_bytes /= 1024.0
    return f"{num_bytes:.1f} TB"


# ── Spark → Polars ────────────────────────────────────────────────────────────


def bench_arrow_spark_to_polars(spark_df):
    """Traditional: collect via Arrow, wrap with Polars."""
    return pl.from_arrow(spark_df.toArrow())


def bench_spiller_spark_to_polars(spark_df):
    """VolumeSpiller: spill to Parquet on UC Volume, scan with Polars."""
    return spill.spark_to_polars(spark_df, cleanup=True, eager=True)


# ── Polars → Spark ────────────────────────────────────────────────────────────


def bench_arrow_polars_to_spark(pl_df):
    """Traditional: serialise to pandas (Arrow-accelerated), createDataFrame."""
    # .count() forces Spark to fully materialise the result, keeping both paths comparable
    return spark.createDataFrame(pl_df.to_pandas()).count()


def bench_spiller_polars_to_spark(pl_df):
    """VolumeSpiller: write Parquet to Volume, read back as Spark DF."""
    return spill.polars_to_spark(pl_df).count()

## Run Benchmarks

This cell will take a few minutes. Each size runs `REPEATS` times — median time is recorded.

In [0]:
results = []  # list of dicts, one per (size, strategy, direction)

for n in SIZES:
    est_size = file_sizes[n]
    print(f"\n{'=' * 60}")
    print(f"  {n:>9,} rows  |  Est. file size: {_format_bytes(est_size)}")
    print(f"{'=' * 60}")

    # Read from pre-materialised Delta table (works on classic and Serverless)
    spark_df = spark.table(temp_tables[n])
    pl_df = make_polars_df(n)

    # ── Spark → Polars ──
    print("  [Spark → Polars]")

    t = _time(lambda: bench_arrow_spark_to_polars(spark_df), REPEATS)
    print(f"    Arrow:         {t['median']:.3f}s  (min {t['min']:.3f}s)")
    results.append({"rows": n, "file_size": est_size, "direction": "Spark→Polars", "strategy": "Arrow", **t})

    t = _time(lambda: bench_spiller_spark_to_polars(spark_df), REPEATS)
    print(f"    VolumeSpiller: {t['median']:.3f}s  (min {t['min']:.3f}s)")
    results.append({"rows": n, "file_size": est_size, "direction": "Spark→Polars", "strategy": "VolumeSpiller", **t})

    # ── Polars → Spark ──
    print("  [Polars → Spark]")

    t = _time(lambda: bench_arrow_polars_to_spark(pl_df), REPEATS)
    print(f"    Arrow:         {t['median']:.3f}s  (min {t['min']:.3f}s)")
    results.append({"rows": n, "file_size": est_size, "direction": "Polars→Spark", "strategy": "Arrow", **t})

    t = _time(lambda: bench_spiller_polars_to_spark(pl_df), REPEATS)
    print(f"    VolumeSpiller: {t['median']:.3f}s  (min {t['min']:.3f}s)")
    results.append({"rows": n, "file_size": est_size, "direction": "Polars→Spark", "strategy": "VolumeSpiller", **t})

print("\nBenchmark complete.")

## Results Table

In [0]:
results_df = pl.DataFrame(results).select(
    "direction",
    "rows",
    "file_size",
    "strategy",
    pl.col("median").round(3).alias("median_s"),
    pl.col("min").round(3).alias("min_s"),
)

# Pivot to side-by-side comparison
pivot = (
    results_df.pivot(on="strategy", index=["direction", "rows", "file_size"], values="median_s")
    .with_columns((pl.col("Arrow") / pl.col("VolumeSpiller")).round(2).alias("Arrow/Spiller ratio"))
    .sort(["direction", "rows"])
)

# Add human-readable file size column
pivot = pivot.with_columns(
    pl.col("file_size").map_elements(_format_bytes, return_dtype=pl.String()).alias("est_file_size")
)

display(pivot.to_pandas())

## Crossover Chart

`Arrow/Spiller ratio > 1` means VolumeSpiller is faster. The horizontal dashed line at 1.0 is the break-even point.

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle("VolumeSpiller vs Arrow: median conversion time", fontsize=14, fontweight="bold")

COLORS = {"Arrow": "#E36414", "VolumeSpiller": "#0F7173"}
directions = ["Spark→Polars", "Polars→Spark"]

for ax, direction in zip(axes, directions):
    subset = results_df.filter(pl.col("direction") == direction)
    for (strategy,), grp in subset.group_by("strategy", maintain_order=True):
        grp_sorted = grp.sort("rows")
        ax.plot(
            grp_sorted["rows"].to_list(),
            grp_sorted["median_s"].to_list(),
            marker="o",
            label=strategy,
            color=COLORS[strategy],
            linewidth=2,
        )
    ax.set_title(direction, fontsize=12)
    ax.set_xlabel("Rows")
    ax.set_ylabel("Time (seconds)")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [0]:
# Ratio chart: values > 1 mean VolumeSpiller is faster
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
fig.suptitle("Speed ratio: Arrow time / VolumeSpiller time  (> 1 = Spiller wins)", fontsize=13, fontweight="bold")

for ax, direction in zip(axes, directions):
    sub = pivot.filter(pl.col("direction") == direction).sort("rows")
    rows = sub["rows"].to_list()
    ratios = sub["Arrow/Spiller ratio"].to_list()

    bar_colors = ["#0F7173" if r >= 1 else "#E36414" for r in ratios]
    ax.bar([f"{r:,}" for r in rows], ratios, color=bar_colors)
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1.2, label="break-even")
    ax.set_title(direction, fontsize=12)
    ax.set_xlabel("Rows")
    ax.set_ylabel("Ratio (Arrow / Spiller)")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## File Size vs Time Analysis

Shows conversion time relative to estimated file size — useful for understanding throughput.

In [0]:
# Throughput: MB/s for each strategy
throughput = results_df.with_columns(
    (pl.col("file_size") / 1024 / 1024).alias("file_size_mb"),
    (pl.col("file_size") / 1024 / 1024 / pl.col("median_s")).round(2).alias("throughput_mb_s"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle("Throughput by file size: MB/s", fontsize=14, fontweight="bold")

for ax, direction in zip(axes, directions):
    subset = throughput.filter(pl.col("direction") == direction)
    for (strategy,), grp in subset.group_by("strategy", maintain_order=True):
        grp_sorted = grp.sort("file_size_mb")
        ax.plot(
            grp_sorted["file_size_mb"].to_list(),
            grp_sorted["throughput_mb_s"].to_list(),
            marker="o",
            label=strategy,
            color=COLORS[strategy],
            linewidth=2,
        )
    ax.set_title(direction, fontsize=12)
    ax.set_xlabel("File Size (MB)")
    ax.set_ylabel("Throughput (MB/s)")
    ax.legend()
    ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## Crossover Point Analysis

In [0]:
print("Crossover Analysis")
print("=" * 70)

for direction in directions:
    sub = pivot.filter(pl.col("direction") == direction).sort("rows")
    rows = sub["rows"].to_list()
    sizes = sub["est_file_size"].to_list()
    ratios = sub["Arrow/Spiller ratio"].to_list()

    crossover = None
    for r, ratio in zip(rows, ratios):
        if ratio >= 1.0:
            crossover = r
            break

    print(f"\n{direction}:")
    print(f"  {'Rows':>12}  {'File Size':>12}  {'Winner':<15}  {'Ratio':>8}")
    print(f"  {'-' * 12}  {'-' * 12}  {'-' * 15}  {'-' * 8}")
    for r, s, ratio in zip(rows, sizes, ratios):
        winner = "VolumeSpiller" if ratio >= 1 else "Arrow"
        print(f"  {r:>12,}  {s:>12}  {winner:<15}  {ratio:>8.2f}x")

    if crossover:
        print(f"\n  ** Crossover: VolumeSpiller faster from {crossover:,} rows onward **")
    else:
        print("\n  ** Arrow is faster at all tested sizes **")

## Cleanup

Drops the temporary Delta tables created for staging and tears down the spill volume.
Set `IS_DEV = False` at the top to also drop the volume.

In [0]:
# Clean up any orphaned tables from previous runs (pattern: benchmark_spark_*)
orphaned = (
    spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA} LIKE 'benchmark_spark_*'")
    .filter(not F.col("isTemporary"))
    .select("tableName")
    .collect()
)

for row in orphaned:
    full_name = f"{CATALOG}.{SCHEMA}.{row['tableName']}"
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    print(f"Dropped orphaned table: {full_name}")

# Clean up current run tables
for table_name in temp_tables.values():
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    print(f"Dropped {table_name}")

# IS_DEV = True → volume is preserved for inspection
# IS_DEV = False → volume is dropped completely (no storage cost)
spill.teardown()